In [1]:
import numpy as np
import pandas as pd
import sympy as sp
from scipy.optimize import differential_evolution

pd.set_option('display.precision', 15)  # Increase decimal precision
pd.set_option('display.width', 150)     # Wider display
pd.set_option('display.max_columns', None)  # Show all column

# Phương pháp Lặp đơn cho Hệ phi tuyến

Điều kiện hội tụ: 

* $G(D) \subset D, \forall X \in D $

* $||J|| \leq q < 1$ --> đảm bảo nghiệm duy nhất ($||\Phi(X)-\Phi(Y)|| \leq q||X-Y||$)

Đánh giá sai số (hậu nghiệm) theo 1 trong 2 công thức sau:

* $\displaystyle ||x_n - x^*|| \leq \frac{q^n}{1-q} ||x_1-x_0||$ $\quad$ (1)

* $\displaystyle ||x_n - x^*|| \leq \frac{q}{1-q} ||x_n-x_{n-1}||$ $\quad$ (2)



# Thuật toán

**Input:** Hàm lặp $\Phi(X)$, miền $D$, sai số $\varepsilon$.

**Output:** Nghiệm xấp xỉ $X^*$.

**Thuật toán:**
1. Nếu $\exists X \in D$ sao cho $\Phi(X) \notin D \Rightarrow$ Dừng (cần chọn $D$ khác).
2. Lập ma trận Jacobi $\displaystyle J(X) = \left[ \frac{\partial\varphi_i}{\partial x_j} \right]$.
3. Tính hệ số co $q$ và chọn chuẩn vector tương ứng:
   - Chuẩn hàng: $\displaystyle q = \max_D ||J||_\infty \Rightarrow$ Dùng chuẩn $||\cdot||_\infty$.
   - Chuẩn cột: $\displaystyle q = \max_D ||J||_1 \Rightarrow$ Dùng chuẩn $||\cdot||_1$.
   - Nếu $q \ge 1 \Rightarrow$ Dừng thuật toán.
4. Chọn $X_0 \in D$, đặt $k=0$.
5. Lặp: Tính $X_{k+1} = \Phi(X_k)$ và sai số $\Delta = ||X_{k+1} - X_k||$.
6. Nếu $\displaystyle \Delta \le \frac{1-q}{q}\varepsilon \Rightarrow X^* = X_{k+1}$, kết thúc. Ngược lại: $k \leftarrow k+1$, quay lại Bước 5.

# Áp dụng

* Các chuẩn vecto: $\displaystyle ||x||_{(1)} = \sum\limits_{i=1}^{k} |x_{i}|$, $\quad$ $\displaystyle ||x||_{(\infty)} = \max\limits_{i} |x_{i}|$
* Các chuẩn ma trận (tính hệ số co):
    * Chuẩn max: $\displaystyle q = m * \max\limits_{i,j} |G_{ij}|$, $\quad$ (chuẩn vecto 1 và chuẩn vecto $\infty$)
    * Chuẩn cột: $\displaystyle q = ||G||_{(1)} = \max\limits_{1\leq j \leq n} \sum\limits_{i=1}^{m} |G_{ij}|$, $\quad$ (chuẩn vecto 1)
    * Chuẩn hàng: $\displaystyle q = ||G||_{(\infty)} = \max\limits_{1\leq i \leq m} \sum\limits_{j=1}^{n} |G_{ij}|$, $\quad$ (chuẩn vecto $\infty$)

## 1. Sai số tuyệt đối

In [2]:
def fixed_point_recursion_absolute (initial_values, q, eps, *funcs, norm):
    n = len(initial_values)
    values = np.array(initial_values)
    results = []
    i=0

    new_eps = eps * (1-q)/ q
    print(f"new_eps: {new_eps}")

    while True:
        new_values = np.array([funcs[j](*values) for j in range(n)])
        
        # Calculate the differences
        diffs = np.abs(new_values - values)
        if (norm == 1): total_diff = np.sum(diffs)
        elif (norm == -1): total_diff = np.max(diffs)
        
        # Append the iteration number, new values, differences, and total_diff to the results list
        results.append([i] + new_values.tolist() + diffs.tolist() + [total_diff])
        values = new_values
        
        
        # Check for convergence
        if total_diff < new_eps:
            break
        i += 1  # Increment the iteration counter
        
    columns = ['k'] + [f'x_({j+1},k+1)' for j in range(n)] + [f'diff_x({j+1},k+1)' for j in range(n)] + ['total_diff']
    df = pd.DataFrame(results, columns=columns)
    print(df.to_string(index=False))  # Print the DataFrame without the index
    
    return values

def check_GD(g_funcs, bounds):
    print("--- CHI TIẾT MIỀN GIÁ TRỊ G(D) ---")
    for i, g_func in enumerate(g_funcs):
        def func_wrapper(vars_array, f=g_func): return float(f(*vars_array))
        def neg_func_wrapper(vars_array, f=g_func): return -float(f(*vars_array))
        g_min = differential_evolution(func_wrapper, bounds).fun
        g_max = -differential_evolution(neg_func_wrapper, bounds).fun
        b_min, b_max = bounds[i]
        print(f"g{i+1}(D): [{g_min:.6f}, {g_max:.6f}] | Giới hạn D{i+1}: [{b_min}, {b_max}]")

def calc_norms(g_funcs, bounds, vars_symbols):
    G_expr = [func(*vars_symbols) for func in g_funcs]
    J = sp.Matrix(G_expr).jacobian(sp.Matrix(vars_symbols))
    J_func = sp.lambdify(vars_symbols, J, 'numpy')
    
    print("\n--- CHI TIẾT TÍNH CHUẨN HÀNG ---")
    row_maxes = []
    for i in range(J.shape[0]):
        def neg_row_i_norm(vars_array, row_idx=i): return -np.sum(np.abs(J_func(*vars_array)[row_idx, :]))
        q_i = -differential_evolution(neg_row_i_norm, bounds).fun
        row_maxes.append(q_i)
        print(f"Max Hàng {i+1} (q{i+1}): {q_i:.4f}")
    row_norm = max(row_maxes)
    print(f"=> Chuẩn hàng q = max({', '.join([f'{q:.4f}' for q in row_maxes])}) = {row_norm:.4f}")
    
    print("\n--- CHI TIẾT TÍNH CHUẨN CỘT ---")
    col_maxes = []
    for j in range(J.shape[1]):
        def neg_col_j_norm(vars_array, col_idx=j): return -np.sum(np.abs(J_func(*vars_array)[:, col_idx]))
        c_j = -differential_evolution(neg_col_j_norm, bounds).fun
        col_maxes.append(c_j)
        print(f"Max Cột {j+1} (c{j+1}): {c_j:.4f}")
    col_norm = max(col_maxes)
    print(f"=> Chuẩn cột q = max({', '.join([f'{c:.4f}' for c in col_maxes])}) = {col_norm:.4f}")
    return row_norm, col_norm

def run_iteration(g_funcs, vars_symbols, initial_values, q, eps, norm=-1):
    G_expr = [func(*vars_symbols) for func in g_funcs]
    G_numpy = sp.lambdify(vars_symbols, G_expr, 'numpy')
    g_funcs_np = [lambda *args, idx=i: G_numpy(*args)[idx] for i in range(len(vars_symbols))]
    
    solution = fixed_point_recursion_absolute(initial_values, q, eps, *g_funcs_np, norm=norm)
    print("Approximate solution:", solution)
    return solution

Bài tập VD trên lớp:

Câu 20a:

In [3]:
def g1(x, y, z):
    return 1/3 * (sp.cos(y*z) + 0.5)

def g2(x, y, z):
    return 1/25 * sp.sqrt(x**2 + 0.3125) - 0.03

def g3(x, y, z):
    return -1/20 * sp.exp(-x*y) - (10*sp.pi-3)/60

g_funcs = [g1, g2, g3]
bounds = [(-1.0, 1.0), (-1.0, 1.0), (-1.0, 1.0)]
check_GD(g_funcs, bounds)

--- CHI TIẾT MIỀN GIÁ TRỊ G(D) ---
g1(D): [0.346767, 0.500000] | Giới hạn D1: [-1.0, 1.0]
g2(D): [-0.007639, 0.015825] | Giới hạn D2: [-1.0, 1.0]
g3(D): [-0.609513, -0.491993] | Giới hạn D3: [-1.0, 1.0]


In [4]:
vars_symbols = sp.symbols('x y z')
row_norm, column_norm = calc_norms(g_funcs, bounds, vars_symbols)


--- CHI TIẾT TÍNH CHUẨN HÀNG ---
Max Hàng 1 (q1): 0.5610
Max Hàng 2 (q2): 0.0349
Max Hàng 3 (q3): 0.2718
=> Chuẩn hàng q = max(0.5610, 0.0349, 0.2718) = 0.5610

--- CHI TIẾT TÍNH CHUẨN CỘT ---
Max Cột 1 (c1): 0.1708
Max Cột 2 (c2): 0.4164
Max Cột 3 (c3): 0.2805
=> Chuẩn cột q = max(0.1708, 0.4164, 0.2805) = 0.4164


In [5]:
initial_values = [0, 0, 0]
eps = 1e-6
q = column_norm
solution = run_iteration(g_funcs, vars_symbols, initial_values, q, eps, norm=1)

new_eps: 1.4015114938959177e-06
 k       x_(1,k+1)          x_(2,k+1)          x_(3,k+1)         diff_x(1,k+1)     diff_x(2,k+1)     diff_x(3,k+1)        total_diff
 0 0.5000000000000 -0.007639320225002 -0.523598775598299 5.000000000000000e-01 0.007639320225002 0.523598775598299 1.031238095823301
 1 0.4999973334186  0.000000000000000 -0.523790123813853 2.666581400245782e-06 0.007639320225002 0.000191348215554 0.007833335021956
 2 0.5000000000000 -0.000000071108732 -0.523598775598299 2.666581400245782e-06 0.000000071108732 0.000191348215554 0.000194085905686
 3 0.5000000000000  0.000000000000000 -0.523598777376017 2.220446049250313e-16 0.000000071108732 0.000000001777718 0.000000072886451
Approximate solution: [ 0.5         0.         -0.52359878]


Câu 20b:

In [ ]:
def g1(x, y, z):
    return (13 - y**2 + 4*z) / 15

def g2(x, y, z):
    return (11 + z - x**2) / 10

def g3(x, y, z):
    return (22 + y**3) / 25

g_funcs = [g1, g2, g3]
bounds = [(0, 1.5), (0, 1.5), (0, 1.5)]
check_GD(g_funcs, bounds)

In [ ]:
vars_symbols = sp.symbols('x y z')
row_norm, column_norm = calc_norms(g_funcs, bounds, vars_symbols)

In [ ]:
initial_values = [0, 0, 0]
eps = 1e-6
q = column_norm
solution = run_iteration(g_funcs, vars_symbols, initial_values, q, eps, norm=1)

Câu 20c:

Câu 20d:

## 2. Sai số tương đối:

In [ ]:
def fixed_point_recursion_relative (initial_values, q, eta, *funcs, norm):
    n = len(initial_values)
    values = np.array(initial_values)
    results = []
    i=0

    new_eta = eta * (1-q)/ q
    print(f"new_eta: {new_eta}")

    while True:
        new_values = np.array([funcs[j](*values) for j in range(n)])
        
        # Calculate the differences
        diffs = np.abs(new_values - values)
        itself = np.abs(new_values)
        if (norm == 1): 
            total_diff = np.sum(diffs)
            total_itself = np.sum(itself)
        elif (norm == -1): 
            total_diff = np.max(diffs)
            total_itself = np.max(itself)
       
        
        relative_diff = total_diff / total_itself
        
        # Append the iteration number, new values, differences, and total_diff to the results list
        results.append([i] + new_values.tolist() + diffs.tolist() + [total_diff] + [relative_diff])
        values = new_values
        
        # Check for convergence
        if i!=0 and relative_diff < new_eta:
            break    

        i += 1  # Increment the iteration counter 
    
    columns = ['k'] + [f'x_({j+1},k+1)' for j in range(n)] + [f'diff_x_({j+1},k+1)' for j in range(n)] + ['total_diff'] + ['relative_diff']
    df = pd.DataFrame(results, columns=columns)
    print(df.to_string(index=False))  # Print the DataFrame without the index
    
    return values

def run_iteration_relative(g_funcs, vars_symbols, initial_values, q, eta, norm=-1):
    G_expr = [func(*vars_symbols) for func in g_funcs]
    G_numpy = sp.lambdify(vars_symbols, G_expr, 'numpy')
    g_funcs_np = [lambda *args, idx=i: G_numpy(*args)[idx] for i in range(len(vars_symbols))]
    
    solution = fixed_point_recursion_relative(initial_values, q, eta, *g_funcs_np, norm=norm)
    print("Approximate solution:", solution)
    return solution

In [ ]:
def g1(x, y):
    return (1/3)*sp.sin(x*y)

def g2(x, y):
    return (1/4)*sp.cos(x**2+y**2)

g_funcs = [g1, g2]
bounds = [(0, 0.33), (0, 0.33)]
check_GD(g_funcs, bounds)

In [ ]:
vars_symbols = sp.symbols('x y')
row_norm, column_norm = calc_norms(g_funcs, bounds, vars_symbols)

In [ ]:
initial_values = [0, 0]
eta = 0.0001
q = column_norm
solution = run_iteration_relative(g_funcs, vars_symbols, initial_values, q, eta, norm=1)